#Mundial 2022

In [ ]:
!pip install ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 21.7 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 wh

##Data 2022

In [ ]:
#UEFA AFC CAF Conmebol  Concacaf
#13   6   5   4         4
nombre_federacion = ['UEFA', 'AFC', 'CAF', 'Conmebol', 'Concacaf', 'Repechaje']
nombre_equipo = ['Alemania', 'Francia', 'Países Bajos', 'Gales', 'Polonia', 'Portugal', 'Croacia',
                 'Inglaterra', 'Bélgica', 'Dinamarca', 'Serbia', 'Suiza', 'España',
                 'Catar', 'Arabia Saudita', 'Corea del Sur', 'Australia', 'Irán', 'Japón',
                 'Ghana', 'Senegal', 'Camerún', 'Marruecos', 'Túnez',
                 'Argentina', 'Brasil', 'Ecuador', 'Uruguay',
                 'Costa Rica', 'Canadá', 'Estados Unidos', 'México']

for i,v in enumerate(nombre_equipo):
  print(i,v)

0 Alemania
1 Francia
2 Países Bajos
3 Gales
4 Polonia
5 Portugal
6 Croacia
7 Inglaterra
8 Bélgica
9 Dinamarca
10 Serbia
11 Suiza
12 España
13 Catar
14 Arabia Saudita
15 Corea del Sur
16 Australia
17 Irán
18 Japón
19 Ghana
20 Senegal
21 Camerún
22 Marruecos
23 Túnez
24 Argentina
25 Brasil
26 Ecuador
27 Uruguay
28 Costa Rica
29 Canadá
30 Estados Unidos
31 México


##Modelo CP

In [ ]:
from ortools.sat.python import cp_model

#crear CSP
model = cp_model.CpModel()

#variables y dominios
##variable g_i_j que significa que el equipo i está en el grupo j
grupos = []
for i in range(32): #para cada equipo
  grupos.append([model.NewBoolVar('grupo_'+str(i)+'_'+str(j)) for j in range(8)])
print(grupos)
##variable de federacion de cada equipo
federacion = [model.NewIntVar(1,6,'federacion_'+str(i)) for i in range(32)]
print(federacion)

##variable de bombos
bombo = [model.NewIntVar(1,4,'bombo_'+str(i)) for i in range(32)]
print(bombo)

#restricciones
#1    2   3   4         5         6
#UEFA AFC CAF Conmebol  Concacaf
#13   6   5   4         4
##distribuir federaciones
for i in range(32):
  if i < 13:
    model.Add(federacion[i] == 1) #UEFA
  elif i < 19:
    model.Add(federacion[i] == 2) #AFC
  elif i < 24:
    model.Add(federacion[i] == 3) #CAF
  elif i < 28:
    model.Add(federacion[i] == 4) #Conmebol
  else:
    model.Add(federacion[i] == 5) #Concacaf

##distribuir bombos
for i in range(32):
  if i in [0,1,2,3,4] + [13] + [23,24]:
    model.Add(bombo[i] == 1)
  elif i in [5,6,7,8,9] + [25] + [27,28]:
    model.Add(bombo[i] == 2)
  elif i in [10,11] + [14,15,16] + [18,19,20]:
    model.Add(bombo[i] == 3)
  else:
    model.Add(bombo[i] == 4)

##cada equipo participa en un solo grupo
for i in range(32):
  model.Add(sum(grupos[i]) == 1)

##cada grupo tiene 4 equipos
for j in range(8):
  model.Add(sum([grupos[i][j] for i in range(32)]) == 4)

##cada grupo tiene 4 equipos de federaciones distintas
grupos_federacion = []
grupos_bombo = []
for i in range(32):
  grupos_federacion.append([model.NewIntVar(0,1000,'gf_'+str(i)+'_'+str(j)) for j in range(8)])
  grupos_bombo.append([model.NewIntVar(0,1000,'gb_'+str(i)+'_'+str(j)) for j in range(8)])
print(grupos_federacion)
print(grupos_bombo)
##calcular las federaciones para los grupos
for i in range(32):
  for j in range(8):
    model.AddMultiplicationEquality(grupos_federacion[i][j],[federacion[i],grupos[i][j]])
    model.AddMultiplicationEquality(grupos_bombo[i][j],[bombo[i],grupos[i][j]])

#ejemplo count_vars(x=[1,2,3,4,1,5,8,1],val=1) -> c = 3
def count_vars(model, x, val, c): #c = cantidad de val en el arreglo x
  b = [model.NewBoolVar('b'+str(i)) for i in range(len(x))]
  for i in range(len(x)):
    #(x[i] == val) <==> b[i]
    ##b[i] => (x[i] == val)
    model.Add(x[i] == val).OnlyEnforceIf(b[i])
    ##(x[i] == val) => b[i] <----> not(b[i]) => not(x[i] == val)
    model.Add(x[i] != val).OnlyEnforceIf(b[i].Not())
  model.Add(c == sum(b))

count = {}
for j in range(8): #8 grupos
  for k in range(1,7): #6 federaciones
    count[(k,j)] = model.NewIntVar(0,1000,'count_'+str(k)+'_'+str(j))
    count_vars(model, [grupos_federacion[i][j] for i in range(32)], k, count[(k,j)])
    if k == 1: #UEFA
      model.Add(count[(k,j)] <= 2)
    elif k > 1: #el resto de federaciones
      model.Add(count[(k,j)] <= 1)

count_bombos = {}
for j in range(8): #8 grupos
  for k in range(1,5): #4 bombos
    count_bombos[(k,j)] = model.NewIntVar(0,1000,'cb_'+str(k)+'_'+str(j))
    count_vars(model, [grupos_bombo[i][j] for i in range(32)], k, count_bombos[(k,j)])
    model.Add(count_bombos[(k,j)] <= 1)

[[grupo_0_0(0..1), grupo_0_1(0..1), grupo_0_2(0..1), grupo_0_3(0..1), grupo_0_4(0..1), grupo_0_5(0..1), grupo_0_6(0..1), grupo_0_7(0..1)], [grupo_1_0(0..1), grupo_1_1(0..1), grupo_1_2(0..1), grupo_1_3(0..1), grupo_1_4(0..1), grupo_1_5(0..1), grupo_1_6(0..1), grupo_1_7(0..1)], [grupo_2_0(0..1), grupo_2_1(0..1), grupo_2_2(0..1), grupo_2_3(0..1), grupo_2_4(0..1), grupo_2_5(0..1), grupo_2_6(0..1), grupo_2_7(0..1)], [grupo_3_0(0..1), grupo_3_1(0..1), grupo_3_2(0..1), grupo_3_3(0..1), grupo_3_4(0..1), grupo_3_5(0..1), grupo_3_6(0..1), grupo_3_7(0..1)], [grupo_4_0(0..1), grupo_4_1(0..1), grupo_4_2(0..1), grupo_4_3(0..1), grupo_4_4(0..1), grupo_4_5(0..1), grupo_4_6(0..1), grupo_4_7(0..1)], [grupo_5_0(0..1), grupo_5_1(0..1), grupo_5_2(0..1), grupo_5_3(0..1), grupo_5_4(0..1), grupo_5_5(0..1), grupo_5_6(0..1), grupo_5_7(0..1)], [grupo_6_0(0..1), grupo_6_1(0..1), grupo_6_2(0..1), grupo_6_3(0..1), grupo_6_4(0..1), grupo_6_5(0..1), grupo_6_6(0..1), grupo_6_7(0..1)], [grupo_7_0(0..1), grupo_7_1(0..1)

##Solver

In [ ]:
solver = cp_model.CpSolver()
status = solver.Solve(model)
if status == cp_model.OPTIMAL:
  #for i in range(32):
    #print(nombre_equipo[i],nombre_federacion[solver.Value(federacion[i])-1])
  #header
  print('\t',end='')
  for i in range(32):
    print(solver.Value(federacion[i]),end='\t')
  print()
  print('\t',end='')
  for i in range(32):
    print(i+1,end='\t')
  print()
  #grupos
  for j in range(8):
    print('G'+str(j+1),end='->\t')
    for i in range(32):
      print(solver.Value(grupos[i][j]),end='\t')
    print()
  print()
  #grupos_federacion
  for j in range(8):
    print('G'+str(j+1),end='->\t')
    for i in range(32):
      print(solver.Value(grupos_federacion[i][j]),end='\t')
    print()
  print()
  #grupos_bombos
  for j in range(8):
    print('G'+str(j+1),end='->\t')
    for i in range(32):
      print(solver.Value(grupos_bombo[i][j]),end='\t')
    print()
  print()
  #mostrar resultado
  for j in range(8):
    print('G'+str(j+1),end='->\t')
    for i in range(32):
      if solver.Value(grupos[i][j]):
        s = nombre_equipo[i] + '(' + nombre_federacion[solver.Value(federacion[i]) - 1] + ')'
        print(s + ' '*(20-len(s)),end='\t')
    [print(solver.Value(count[(k,j)]),end=' ') for k in range(1,7)]
    print(end='\t')
    [print(solver.Value(count_bombos[(k,j)]),end=' ') for k in range(1,5)]
    print()

	1	1	1	1	1	1	1	1	1	1	1	1	1	2	2	2	2	2	2	3	3	3	3	3	4	4	4	4	5	5	5	5	
	1	2	3	4	5	6	7	8	9	10	11	12	13	14	15	16	17	18	19	20	21	22	23	24	25	26	27	28	29	30	31	32	
G1->	1	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	1	0	0	0	0	0	0	0	0	0	
G2->	0	0	0	0	0	1	0	0	0	0	0	0	0	1	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	1	0	0	
G3->	0	0	0	0	0	0	0	0	0	0	0	1	1	0	0	0	0	0	0	0	0	0	0	1	0	0	0	1	0	0	0	0	
G4->	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	1	0	1	0	0	0	
G5->	0	1	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	1	
G6->	0	0	0	1	0	0	0	0	1	0	0	0	0	0	0	0	0	1	0	1	0	0	0	0	0	0	0	0	0	0	0	0	
G7->	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	1	0	0	0	0	1	0	0	1	0	0	0	0	0	0	0	
G8->	0	0	0	0	1	0	0	1	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	

G1->	1	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	3	0	0	0	0	0	0	0	0	0	
G2->	0	0	0	0	0	1	0	0	0	0	0	0	0	2	0	0	0	0	0	0	3	0	0	0	0	0	0	0	0	5	0	0	
G3->	0	0	0	0	0	0	0	0	0	0	0	1	1	0	0	0	0	0	0	0	0	0	0	3	0	0	0	4	0	0	0	0	
G4->	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	4	0	5	0	0	0	
G5->

##All solutions

In [ ]:
class VarArraySolutionPrinter(cp_model.CpSolverSolutionCallback):
  def __init__(self, variables):
    cp_model.CpSolverSolutionCallback.__init__(self)
    self.__variables = variables
    self.__solution_count = 0

  def on_solution_callback(self):
    self.__solution_count += 1
    for j in range(8):
      print('G'+str(j+1),end='->\t')
      for i in range(32):
        if self.Value(self.__variables[i][j]):
          s = nombre_equipo[i]
          print(s+' '*(20-len(s)),end='\t')
      print()
    print()

  def solution_count(self):
    return self.__solution_count

In [ ]:
solver = cp_model.CpSolver()
solution_printer = VarArraySolutionPrinter(grupos)
# Enumerate all solutions.
solver.parameters.enumerate_all_solutions = True
# Solve.
status = solver.Solve(model, solution_printer)

Se truncaron las últimas líneas 5000 del resultado de transmisión.
G5->	España              	Corea del Sur       	Camerún             	Uruguay             	
G6->	Bélgica             	Serbia              	Arabia Saudita      	México              	
G7->	Inglaterra          	UEFA1               	Irán                	EEUU                	
G8->	Francia             	Alemania            	Senegal             	OFC/Concacaf        	

G1->	Portugal            	Croacia             	Japón               	AFC/Conmebol        	
G2->	Suiza               	Marruecos           	Argentina           	Canadá              	
G3->	Dinamarca           	Polonia             	Catar               	Ghana               	
G4->	Países Bajos        	Arabia Saudita      	Túnez               	Brasil              	
G5->	España              	Corea del Sur       	Camerún             	Uruguay             	
G6->	Francia             	Serbia              	Ecuador             	México              	
G7->	Inglaterra          	UEFA1 

#Mundial 2026

In [ ]:
#Complete el mismo modelo para los equipos del mundial 2026

#UEFA AFC CAF Conmebol  Concacaf
#13   6   5   4         4
nombre_federacion = ['UEFA', 'AFC', 'CAF', 'Conmebol', 'Concacaf', 'OFC']
nombre_equipo = ['Alemania', 'Inglaterra', 'Portugal', 'República Checa', 'Croacia', 'Austria', 'Bélgica', 'Suecia', 'Bosnia y Herzegovina', 'Escocia', 'Suiza', 'España', 'Noruega', 'Turquía', 'Francia', 'Países Bajos',
'Arabia Saudita', 'Corea del Sur', 'Argelia', 'Irak', 'Irán', 'Australia', 'Japón', 'Jordania', 'Catar', 'Uzbekistán',
'Costa de Marfil', 'R. D. del Congo', 'Senegal', 'Sudáfrica', 'Egipto', 'Marruecos', 'Túnez', 'Cabo Verde', 'Ghana',
'Colombia', 'Paraguay', 'Argentina', 'Ecuador', 'Brasil', 'Uruguay',
'Haití', 'Curazao', 'México', 'Estados Unidos', 'Canadá', 'Panamá',
'Nueva Zelanda']

for i,v in enumerate(nombre_equipo):
  print(i,v)

0 Alemania
1 Inglaterra
2 Portugal
3 República Checa
4 Croacia
5 Austria
6 Bélgica
7 Suecia
8 Bosnia y Herzegovina
9 Escocia
10 Suiza
11 España
12 Noruega
13 Turquía
14 Francia
15 Países Bajos
16 Arabia Saudita
17 Corea del Sur
18 Argelia
19 Irak
20 Irán
21 Australia
22 Japón
23 Jordania
24 Catar
25 Uzbekistán
26 Costa de Marfil
27 R. D. del Congo
28 Senegal
29 Sudáfrica
30 Egipto
31 Marruecos
32 Túnez
33 Cabo Verde
34 Ghana
35 Colombia
36 Paraguay
37 Argentina
38 Ecuador
39 Brasil
40 Uruguay
41 Haití
42 Curazao
43 México
44 Estados Unidos
45 Canadá
46 Panamá
47 Nueva Zelanda


In [ ]:
from ortools.sat.python import cp_model

#crear CSP
model = cp_model.CpModel()

n = len(nombre_equipo)
#variables y dominios
##variable g_i_j que significa que el equipo i está en el grupo j
grupos = []
for i in range(n): #para cada equipo
  grupos.append([model.NewBoolVar('grupo_'+str(i)+'_'+str(j)) for j in range(12)])
print(grupos)
##variable de federacion de cada equipo
federacion = [model.NewIntVar(1,6,'federacion_'+str(i)) for i in range(n)]
print(federacion)

##variable de bombos
bombo = [model.NewIntVar(1,4,'bombo_'+str(i)) for i in range(n)]
print(bombo)

#restricciones
#1    2   3   4         5         6
#UEFA AFC CAF Conmebol  Concacaf
#13   6   5   4         4
##distribuir federaciones
for i in range(n):
  if i < 16:
    model.Add(federacion[i] == 1) #UEFA
  elif i < 26:
    model.Add(federacion[i] == 2) #AFC
  elif i < 35:
    model.Add(federacion[i] == 3) #CAF
  elif i < 41:
    model.Add(federacion[i] == 4) #Conmebol
  elif i < 47:
    model.Add(federacion[i] == 5) #Concacaf
  else:
    model.Add(federacion[i] == 6) #OFC

##distribuir bombos
for i in range(n):
  if i in [43,44,45,11,37,1,14,39,2,15,0,6]:
    model.Add(bombo[i] == 1)
  elif i in [4,35,40,31,10,22,28,20,17,38,5,21]:
    model.Add(bombo[i] == 2)
  elif i in [46,30,18,12,9,36,26,32,16,24,25,29]:
    model.Add(bombo[i] == 3)
#  else:
#    model.Add(bombo[i] == 4)

##cada equipo participa en un solo grupo
for i in range(n):
  model.Add(sum(grupos[i]) == 1)

##cada grupo tiene 4 equipos
for j in range(12):
  model.Add(sum([grupos[i][j] for i in range(n)]) == 4)

##cada grupo tiene 4 equipos de federaciones distintas
grupos_federacion = []
grupos_bombo = []
for i in range(n):
  grupos_federacion.append([model.NewIntVar(0,1000,'gf_'+str(i)+'_'+str(j)) for j in range(12)])
  grupos_bombo.append([model.NewIntVar(0,1000,'gb_'+str(i)+'_'+str(j)) for j in range(12)])
print(grupos_federacion)
print(grupos_bombo)
##calcular las federaciones para los grupos
for i in range(n):
  for j in range(12):
    model.AddMultiplicationEquality(grupos_federacion[i][j],[federacion[i],grupos[i][j]])
    model.AddMultiplicationEquality(grupos_bombo[i][j],[bombo[i],grupos[i][j]])

#ejemplo count_vars(x=[1,2,3,4,1,5,8,1],val=1) -> c = 3
def count_vars(model, x, val, c): #c = cantidad de val en el arreglo x
  b = [model.NewBoolVar('b'+str(i)) for i in range(len(x))]
  for i in range(len(x)):
    #(x[i] == val) <==> b[i]
    ##b[i] => (x[i] == val)
    model.Add(x[i] == val).OnlyEnforceIf(b[i])
    ##(x[i] == val) => b[i] <----> not(b[i]) => not(x[i] == val)
    model.Add(x[i] != val).OnlyEnforceIf(b[i].Not())
  model.Add(c == sum(b))

count = {}
for j in range(12): #12 grupos
  for k in range(1,7): #6 federaciones
    count[(k,j)] = model.NewIntVar(0,1000,'count_'+str(k)+'_'+str(j))
    count_vars(model, [grupos_federacion[i][j] for i in range(32)], k, count[(k,j)])
    if k == 1: #UEFA
      model.Add(count[(k,j)] <= 2)
    elif k > 1: #el resto de federaciones
      model.Add(count[(k,j)] <= 1)

count_bombos = {}
for j in range(12): #8 grupos
  for k in range(1,5): #4 bombos
    count_bombos[(k,j)] = model.NewIntVar(0,1000,'cb_'+str(k)+'_'+str(j))
    count_vars(model, [grupos_bombo[i][j] for i in range(32)], k, count_bombos[(k,j)])
    model.Add(count_bombos[(k,j)] <= 1)

[[grupo_0_0(0..1), grupo_0_1(0..1), grupo_0_2(0..1), grupo_0_3(0..1), grupo_0_4(0..1), grupo_0_5(0..1), grupo_0_6(0..1), grupo_0_7(0..1), grupo_0_8(0..1), grupo_0_9(0..1), grupo_0_10(0..1), grupo_0_11(0..1)], [grupo_1_0(0..1), grupo_1_1(0..1), grupo_1_2(0..1), grupo_1_3(0..1), grupo_1_4(0..1), grupo_1_5(0..1), grupo_1_6(0..1), grupo_1_7(0..1), grupo_1_8(0..1), grupo_1_9(0..1), grupo_1_10(0..1), grupo_1_11(0..1)], [grupo_2_0(0..1), grupo_2_1(0..1), grupo_2_2(0..1), grupo_2_3(0..1), grupo_2_4(0..1), grupo_2_5(0..1), grupo_2_6(0..1), grupo_2_7(0..1), grupo_2_8(0..1), grupo_2_9(0..1), grupo_2_10(0..1), grupo_2_11(0..1)], [grupo_3_0(0..1), grupo_3_1(0..1), grupo_3_2(0..1), grupo_3_3(0..1), grupo_3_4(0..1), grupo_3_5(0..1), grupo_3_6(0..1), grupo_3_7(0..1), grupo_3_8(0..1), grupo_3_9(0..1), grupo_3_10(0..1), grupo_3_11(0..1)], [grupo_4_0(0..1), grupo_4_1(0..1), grupo_4_2(0..1), grupo_4_3(0..1), grupo_4_4(0..1), grupo_4_5(0..1), grupo_4_6(0..1), grupo_4_7(0..1), grupo_4_8(0..1), grupo_4_9(0..

In [ ]:
solver = cp_model.CpSolver()
status = solver.Solve(model)
if status == cp_model.OPTIMAL:
  #for i in range(n):
    #print(nombre_equipo[i],nombre_federacion[solver.Value(federacion[i])-1])
  #header
  print('\t',end='')
  for i in range(n):
    print(solver.Value(federacion[i]),end='\t')
  print()
  print('\t',end='')
  for i in range(n):
    print(i+1,end='\t')
  print()
  #grupos
  for j in range(12):
    print('G'+str(j+1),end='->\t')
    for i in range(n):
      print(solver.Value(grupos[i][j]),end='\t')
    print()
  print()
  #grupos_federacion
  for j in range(12):
    print('G'+str(j+1),end='->\t')
    for i in range(n):
      print(solver.Value(grupos_federacion[i][j]),end='\t')
    print()
  print()
  #grupos_bombos
  for j in range(12):
    print('G'+str(j+1),end='->\t')
    for i in range(n):
      print(solver.Value(grupos_bombo[i][j]),end='\t')
    print()
  print()
  #mostrar resultado
  for j in range(12):
    print('G'+str(j+1),end='->\t')
    for i in range(n):
      if solver.Value(grupos[i][j]):
        s = nombre_equipo[i] + ' (' + nombre_federacion[solver.Value(federacion[i]) - 1] + ')'
        print(s + ' '*(20-len(s)),end='\t')
    #[print(solver.Value(count[(k,j)]),end=' ') for k in range(1,7)]
    #print(end='\t')
    #[print(solver.Value(count_bombos[(k,j)]),end=' ') for k in range(1,5)]
    print()

	1	1	1	1	1	1	1	1	1	1	1	1	1	1	1	1	2	2	2	2	2	2	2	2	2	2	3	3	3	3	3	3	3	3	3	4	4	4	4	4	4	5	5	5	5	5	5	6	
	1	2	3	4	5	6	7	8	9	10	11	12	13	14	15	16	17	18	19	20	21	22	23	24	25	26	27	28	29	30	31	32	33	34	35	36	37	38	39	40	41	42	43	44	45	46	47	48	
G1->	0	0	0	0	0	0	0	0	0	0	0	1	0	1	0	0	0	0	0	0	0	0	0	0	0	1	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	
G2->	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	
G3->	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	
G4->	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	1	0	0	0	0	1	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	
G5->	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	1	0	0	0	0	0	0	1	1	0	0	0	0	0	0	0	0	0	
G6->	1	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	
G7->	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	
G8->	0	0	0	1	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	